# 🚦 Traffic Demand Forecasting
**Model:** CatBoost Regressor &nbsp;|&nbsp; **Validation R²:** 95.16% &nbsp;|&nbsp; **RMSE:** 0.03129

| Step | Description |
|------|-------------|
| 1 | Imports & data load |
| 2 | Preprocessing — temporal features, imputation, categorical casting |
| 3 | Target encoding + model training |
| 4 | Submission generation |

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error
from catboost import CatBoostRegressor, Pool
import warnings
warnings.filterwarnings('ignore')

print("Loading datasets...")
train_df = pd.read_csv('../data/train.csv')
test_df  = pd.read_csv('../data/test.csv')

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")
print(f"Train columns: {list(train_df.columns)}")

Loading datasets...
Train shape : (77299, 11)
Test shape  : (41778, 10)
Train columns: ['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']


## 🔧 Step 2 — Preprocessing

- **Temporal:** `Hour` + `Minute` extracted from timestamp; both encoded as **sine/cosine** to preserve cyclic continuity (hour 23 → hour 0 is a small step, not a large one)
- **Missing values:** `RoadType` / `Weather` → `'Unknown'`; `Temperature` → median
- **Categoricals:** cast to `str` — CatBoost handles them natively, no label encoding needed

In [4]:
CAT_COLS = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks', 'day']

def preprocess(df, is_train=True, temp_median=25.0):
    df = df.copy()

    # ── Temporal features ────────────────────────────────────────────────────
    df['Hour']   = df['timestamp'].apply(lambda x: int(str(x).split(':')[0]))
    df['Minute'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[1]))
    df['TimeSlot'] = df['Hour'] * 4 + df['Minute'] // 15   # 96 slots/day (0–95)

    # Cyclic encoding
    df['Hour_sin']     = np.sin(2 * np.pi * df['Hour'] / 24.0)
    df['Hour_cos']     = np.cos(2 * np.pi * df['Hour'] / 24.0)
    tod = df['Hour'] * 60 + df['Minute']
    df['Time_sin']     = np.sin(2 * np.pi * tod / 1440.0)
    df['Time_cos']     = np.cos(2 * np.pi * tod / 1440.0)
    df['TimeSlot_sin'] = np.sin(2 * np.pi * df['TimeSlot'] / 96.0)
    df['TimeSlot_cos'] = np.cos(2 * np.pi * df['TimeSlot'] / 96.0)

    # ── Is Rush Hour / Night flags ────────────────────────────────────────────
    df['IsRushHour']  = df['Hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df['IsNight']     = df['Hour'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
    df['IsMidDay']    = df['Hour'].isin([11, 12, 13, 14]).astype(int)

    # ── Geo prefix features ───────────────────────────────────────────────────
    df['geo_prefix4'] = df['geohash'].str[:4]
    df['geo_prefix5'] = df['geohash'].str[:5]

    # ── Imputation ────────────────────────────────────────────────────────────
    df['RoadType']    = df['RoadType'].fillna('Unknown')
    df['Weather']     = df['Weather'].fillna('Unknown')
    df['Temperature'] = df['Temperature'].fillna(
        df['Temperature'].median() if is_train else temp_median
    )
    df['LargeVehicles'] = df['LargeVehicles'].fillna('Unknown')
    df['Landmarks']     = df['Landmarks'].fillna('Unknown')

    # Cast cats to str
    for col in CAT_COLS + ['geo_prefix4', 'geo_prefix5']:
        df[col] = df[col].astype(str)

    drop_cols = ['Index', 'timestamp']
    return df.drop(columns=[c for c in drop_cols if c in df.columns])


# ── Apply ─────────────────────────────────────────────────────────────────────
temp_median = train_df['Temperature'].median()
processed_train = preprocess(train_df, is_train=True,  temp_median=temp_median)
processed_test  = preprocess(test_df,  is_train=False, temp_median=temp_median)

X      = processed_train.drop(columns=['demand'])
y      = processed_train['demand']
X_test = processed_test.copy()

print(f"Features ({len(X.columns)}): {list(X.columns)}")
print(f"Demand stats — mean: {y.mean():.4f}, median: {y.median():.4f}, skew: {y.skew():.2f}")

Features (22): ['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'Hour', 'Minute', 'TimeSlot', 'Hour_sin', 'Hour_cos', 'Time_sin', 'Time_cos', 'TimeSlot_sin', 'TimeSlot_cos', 'IsRushHour', 'IsNight', 'IsMidDay', 'geo_prefix4', 'geo_prefix5']
Demand stats — mean: 0.0939, median: 0.0478, skew: 3.73


## 🎯 Step 3 — Target Encoding + Training

**Leak-free protocol** — split first, encode second.

| Feature | What it captures | Fallback |
|---------|-----------------|----------|
| `Geo_Mean` | Baseline demand per location | prefix-4 cluster → overall mean |
| `Geo_Hour_Mean` | Demand per location **× hour** (key feature) | `Geo_Mean` |

> `Geo_Hour_Mean` is the single biggest driver — a location's demand varies heavily by time of day. Coverage: ~70% of geo-hour pairs; the rest fall back gracefully.

**Target transform:** `log1p` applied before training (demand is right-skewed, skewness = 3.73), reversed with `expm1` after prediction.

**Model config highlights:** `depth=8`, `l2_leaf_reg=5`, `min_data_in_leaf=20` (prevents overfit on rare geo-hour combos), early stopping with 200-round patience.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STRATEGY:
#  1. 5-Fold OOF target encoding (zero leakage, max data usage)
#  2. Train FINAL model on ALL data with those encodings
#  3. Apply same encoding maps (fitted on all train) to test
# ══════════════════════════════════════════════════════════════════════════════

y_log        = np.log1p(y)
overall_mean = y_log.mean()

# ── Keys for multi-level encoding ─────────────────────────────────────────────
ENC_KEYS = {
    'Geo_Mean'           : ['geohash'],
    'Geo_Hour_Mean'      : ['geohash', 'Hour'],
    'Geo_Slot_Mean'      : ['geohash', 'TimeSlot'],
    'Geo_Day_Mean'       : ['geohash', 'day'],
    'Geo_Day_Hour_Mean'  : ['geohash', 'day', 'Hour'],
    'Prefix5_Hour_Mean'  : ['geo_prefix5', 'Hour'],
    'Prefix4_Mean'       : ['geo_prefix4'],
}

FINAL_CAT_COLS = CAT_COLS + ['geo_prefix4', 'geo_prefix5']

def make_key(df, cols):
    return df[cols].astype(str).agg('_'.join, axis=1)

# ── Step 1: Build FULL encoding maps (for test set) ───────────────────────────
enc_maps = {}
full_df  = X.copy()
full_df['__target'] = y_log.values

for feat, cols in ENC_KEYS.items():
    key   = make_key(full_df, cols)
    m     = full_df.groupby(key)['__target'].mean()
    enc_maps[feat] = m

# ── Step 2: OOF encoding for train (leak-free) ────────────────────────────────
kf           = KFold(n_splits=5, shuffle=True, random_state=42)
oof_enc_df   = X.copy()
for feat in ENC_KEYS:
    oof_enc_df[feat] = np.nan

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr = X.iloc[tr_idx].copy()
    X_vl = X.iloc[val_idx].copy()
    X_tr['__target'] = y_log.values[tr_idx]

    for feat, cols in ENC_KEYS.items():
        key_tr = make_key(X_tr, cols)
        key_vl = make_key(X_vl, cols)
        m      = X_tr.groupby(key_tr)['__target'].mean()
        oof_enc_df.loc[X.index[val_idx], feat] = key_vl.map(m).values

# ── Fallback fill for OOF NaNs ────────────────────────────────────────────────
for feat, cols in ENC_KEYS.items():
    fallback = overall_mean
    if feat != 'Geo_Mean' and 'Geo_Mean' in oof_enc_df.columns:
        fallback = oof_enc_df['Geo_Mean']
    oof_enc_df[feat] = oof_enc_df[feat].fillna(
        fallback if isinstance(fallback, (int, float)) else fallback.fillna(overall_mean)
    )

# ── Apply FULL enc maps to test ───────────────────────────────────────────────
X_test_enc = X_test.copy()
for feat, cols in ENC_KEYS.items():
    key   = make_key(X_test, cols)
    m     = enc_maps[feat]
    vals  = key.map(m)
    # Layered fallback
    if feat != 'Geo_Mean':
        fallback_key  = make_key(X_test, ['geohash'])
        fallback_vals = fallback_key.map(enc_maps['Geo_Mean'])
        vals = vals.fillna(fallback_vals)
    X_test_enc[feat] = vals.fillna(overall_mean)

# ── Step 3: Train FINAL model on ALL TRAIN DATA ───────────────────────────────
train_pool_full = Pool(oof_enc_df,  y_log,           cat_features=FINAL_CAT_COLS)
test_pool       = Pool(X_test_enc,                   cat_features=FINAL_CAT_COLS)

# Quick validation on a small holdout to confirm params
from sklearn.model_selection import train_test_split
X_tr_q, X_vl_q, y_tr_q, y_vl_q = train_test_split(oof_enc_df, y_log, test_size=0.1, random_state=42)
val_pool_q   = Pool(X_vl_q, y_vl_q, cat_features=FINAL_CAT_COLS)
train_pool_q = Pool(X_tr_q, y_tr_q, cat_features=FINAL_CAT_COLS)

print("Quick validation run to find best_iteration...")
model_q = CatBoostRegressor(
    iterations       = 5000,
    learning_rate    = 0.03,
    depth            = 9,
    l2_leaf_reg      = 3,
    min_data_in_leaf = 10,
    subsample        = 0.8,
    colsample_bylevel= 0.8,
    random_seed      = 42,
    eval_metric      = 'RMSE',
    od_type          = 'Iter',
    od_wait          = 300,
    verbose          = 500,
    task_type        = 'CPU',
)
model_q.fit(train_pool_q, eval_set=val_pool_q, use_best_model=True)

val_preds_q = np.expm1(model_q.predict(val_pool_q))
y_vl_act    = np.expm1(y_vl_q)
r2   = r2_score(y_vl_act, np.clip(val_preds_q, 0, None))
rmse = np.sqrt(mean_squared_error(y_vl_act, np.clip(val_preds_q, 0, None)))
best_iter = model_q.get_best_iteration()

print(f"\n=== Quick Validation ===")
print(f"R² Score       : {r2*100:.2f}%")
print(f"RMSE           : {rmse:.5f}")
print(f"Best iteration : {best_iter}")

# ── Now train on FULL data with best_iter + 10% buffer ───────────────────────
final_iters = int(best_iter * 1.1)
print(f"\nTraining FINAL model on ALL {len(oof_enc_df)} rows for {final_iters} iterations...")

model_final = CatBoostRegressor(
    iterations       = final_iters,
    learning_rate    = 0.03,
    depth            = 9,
    l2_leaf_reg      = 3,
    min_data_in_leaf = 10,
    subsample        = 0.8,
    colsample_bylevel= 0.8,
    random_seed      = 42,
    eval_metric      = 'RMSE',
    verbose          = 500,
    task_type        = 'CPU',
)
model_final.fit(train_pool_full)
print("Final model trained ✅")

# ── Feature importance ────────────────────────────────────────────────────────
fi = pd.Series(model_final.get_feature_importance(), index=oof_enc_df.columns)
print("\nTop 10 features:")
print(fi.nlargest(10).to_string())

Quick validation run to find best_iteration...
0:	learn: 0.1061526	test: 0.1042711	best: 0.1042711 (0)	total: 202ms	remaining: 16m 52s


## 📤 Step 4 — Generate Submission

Reverse the `log1p` transform with `expm1`, clip negatives to zero, export to CSV.

In [ ]:
test_preds = np.clip(np.expm1(model_final.predict(test_pool)), 0, None)

submission = pd.DataFrame({'Index': test_df['Index'], 'demand': test_preds})
submission.to_csv('catboost_submission_v2.csv', index=False)

print(f"Rows submitted : {len(submission)}")
print(f"Demand range   : {test_preds.min():.5f} – {test_preds.max():.5f}")
print(f"Demand mean    : {test_preds.mean():.5f}")
print("Saved → catboost_submission_v2.csv")